<a href="https://colab.research.google.com/github/AsianUser/MusicBCI/blob/Cecilia/Machine_Learning_Model_MusicBCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### i pasted sample code, have to edit for our specific csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import timm

import matplotlib.pyplot as plt # For data viz
import pandas as pd
import numpy as np
import sys
from tqdm.notebook import tqdm

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, eeg_data, labels, transform=None):
        """
        Args:
            eeg_data (np.array): A numpy array of EEG data (e.g., shape: [N_samples, N_channels, N_timepoints]).
            labels (np.array): A numpy array of labels (e.g., shape: [N_samples]).
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.eeg_data = torch.from_numpy(eeg_data).float()
        self.labels = torch.from_numpy(labels).long() # Use .long() for classification labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        data = self.eeg_data[idx]
        label = self.labels[idx]

        if self.transform:
            data = self.transform(data)

        return data, label


# Example of creating the dataset (replace with your actual data loading logic)
# Assuming you have data in numpy arrays
# Example shapes: eeg_data.shape (1000, 32, 128), labels.shape (1000,)
# data, labels = load_your_eeg_data_and_labels()
# dataset = EEGDataset(data, labels)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
dataset = EEGDataset(data_dir = '')
len(dataset)   #just so we know the length

In [ ]:
# Get a dictionary associating target values with folder names
data_dir = ''
target_to_class = {v: k for k, v in "FolderXXXX"(data_dir).class_to_idx.items()}
print(target_to_class)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

data_dir = ''
dataset = EEGDataset(data_dir, transform)

In [ ]:
# iterate over dataset
for label in dataset:
    break

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class EEG_CNN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(EEG_CNN, self).__init__()
        # Define 1D convolutional layers
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=5, stride=1, padding=2)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2)

        # Calculate the size of the flattened layer (depends on input time dimension)
        # This is a placeholder; actual size needs adjustment based on your data shape and conv/pool layers
        self.fc_input_features = self._get_conv_output((input_channels, 128))

        # Define fully connected layers
        self.fc1 = nn.Linear(self.fc_input_features, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def _get_conv_output(self, shape):
        # Helper function to calculate the output size after conv/pool layers
        # This can be tricky; often it's easier to run a dummy tensor once to find the shape
        dummy_input = torch.rand(1, shape[0], shape[1])
        x = self.conv1(dummy_input)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.pool(x)
        return int(np.prod(x.size()))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, self.fc_input_features) # Flatten the output
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1) # Use log_softmax for NLLLoss

# Example of initializing the model
# INPUT_CHANNELS = 32 # number of EEG channels
# NUM_CLASSES = 5   # number of distinct labels
# model = EEG_CNN(INPUT_CHANNELS, NUM_CLASSES)

In [ ]:
import torch.optim as optim

def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(dataloader):
            # Move data to GPU if available
            # inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(dataloader):.4f}')

# Example of training setup
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)
# train_model(model, dataloader, criterion, optimizer)